# 05 - Índice de Riesgo Climático

Este notebook construye el índice de riesgo climático histórico del seguro de café.

**Objetivo del notebook**

A partir de las features intra-anuales con clúster y las variables seleccionadas:

1. Ajustar un modelo Ridge por clúster.
2. Calcular un índice climático calibrado.
3. Definir evento de pérdida.
4. Optimizar el trigger del seguro.
5. Construir la función de pago.
6. Calcular prima pura y prima comercial como tasas.
7. Guardar artefactos necesarios para simulación, modelo completo y API.

Este notebook no calcula valores monetarios finales con precio del café.  
El precio se incorporará posteriormente en el modelo completo del seguro.


## 1. Librerías

In [ ]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, auc


## 2. Rutas y parámetros generales

In [ ]:
# ============================================================
# RUTAS ROBUSTAS DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH_MODEL = PROJECT_ROOT / "data" / "model"
PATH_OUTPUTS = PROJECT_ROOT / "data" / "outputs"

PATH_MODEL.mkdir(parents=True, exist_ok=True)
PATH_OUTPUTS.mkdir(parents=True, exist_ok=True)

# Entradas
FEATURES_FILE = PATH_MODEL / "features_intra_anuales_2007-2024_clusters.csv"
VARIABLES_FILE = PATH_MODEL / "variables_seleccionadas.csv"
IMPORTANCIA_RF_FILE = PATH_MODEL / "importancia_variables_rf.csv"

# Salidas modelo
OUTPUT_COEF = PATH_MODEL / "coeficientes_indice_por_cluster.csv"
OUTPUT_METRICAS = PATH_MODEL / "metricas_modelo_indice_por_cluster.csv"
OUTPUT_PARAMETROS = PATH_MODEL / "parametros_seguro.csv"
OUTPUT_VARIABLES_INDICE = PATH_MODEL / "variables_indice_final.csv"

# Salidas de validación / seguimiento
OUTPUT_INDICE = PATH_OUTPUTS / "indice_riesgo_climatico_2007-2024.csv"
OUTPUT_RESUMEN_RIESGO = PATH_OUTPUTS / "resumen_riesgo_indice.csv"
OUTPUT_EVAL_TRIGGER = PATH_OUTPUTS / "evaluacion_trigger_indice.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURES_FILE:", FEATURES_FILE)
print("VARIABLES_FILE:", VARIABLES_FILE)


In [ ]:
# ============================================================
# PARÁMETROS DEL MODELO
# ============================================================

TARGET = "Rendimiento (t/ha)"

# Evento de pérdida: rendimiento por debajo de este percentil
LOSS_QUANTILE = 0.30

# Penalización relativa del falso negativo en la selección del trigger
# FN = no pagar cuando sí hubo pérdida
PENALTY_FN = 2.0

# Exponente de la función de pago
# 1.0 = lineal; >1 suaviza pagos intermedios
PAYOUT_EXPONENT = 1.30

# Margen comercial sobre prima pura
MARGEN_COMERCIAL = 0.30

# Máximo de variables del índice para mantener simplicidad e interpretabilidad
MAX_VARIABLES_INDICE = 12


## 3. Carga y validación de datos

In [ ]:
df_model = pd.read_csv(FEATURES_FILE)

cols_base = ["municipio", "anio", "cluster", TARGET]
faltantes = [c for c in cols_base if c not in df_model.columns]

if faltantes:
    raise ValueError(f"Faltan columnas base necesarias: {faltantes}")

df_model["cluster"] = df_model["cluster"].astype(int)

print("Filas:", len(df_model))
print("Municipios:", df_model["municipio"].nunique())
print("Años:", df_model["anio"].min(), "-", df_model["anio"].max())
print("\nDistribución por clúster:")
print(df_model["cluster"].value_counts().sort_index())


## 4. Variables del índice

In [ ]:
variables_sel = pd.read_csv(VARIABLES_FILE)["variable"].dropna().tolist()

# Variables que no deben entrar al índice por fuga de información, identidad o estructura.
vars_excluir = {
    TARGET,
    "produccion_t",
    "Área sembrada (ha)",
    "area_sembrada",
    "municipio",
    "date",
    "anio",
    "mes",
    "cluster"
}

vars_indice = [
    v for v in variables_sel
    if v in df_model.columns
    and v not in vars_excluir
    and pd.api.types.is_numeric_dtype(df_model[v])
]

# Ordenar por importancia RF si existe el archivo.
if IMPORTANCIA_RF_FILE.exists():
    importancia_rf = pd.read_csv(IMPORTANCIA_RF_FILE, index_col=0)

    # Puede venir con una columna sin nombre o con el valor como primera columna.
    if importancia_rf.shape[1] >= 1:
        importancia_rf = importancia_rf.iloc[:, 0]

    orden_rf = [v for v in importancia_rf.sort_values(ascending=False).index if v in vars_indice]
    vars_indice = orden_rf + [v for v in vars_indice if v not in orden_rf]

# Limitar número de variables para mantener simplicidad operacional.
vars_indice = vars_indice[:MAX_VARIABLES_INDICE]

if len(vars_indice) == 0:
    raise ValueError("No hay variables válidas para construir el índice.")

pd.DataFrame({"variable": vars_indice}).to_csv(OUTPUT_VARIABLES_INDICE, index=False)

print(f"Variables finales del índice ({len(vars_indice)}):")
for v in vars_indice:
    print("-", v)


## 5. Modelo Ridge por clúster

Se ajusta un modelo Ridge independiente por clúster para respetar los distintos regímenes climáticos.

La salida principal de este bloque son:

- coeficientes por clúster
- métricas por clúster
- rendimiento predicho histórico


In [ ]:
coeficientes = {}
metricas = []

df_model["rendimiento_predicho"] = np.nan

for c in sorted(df_model["cluster"].unique()):

    df_c = df_model[df_model["cluster"] == c].copy()

    X_c = df_c[vars_indice]
    y_c = df_c[TARGET]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    n_obs = len(df_c)
    cv = min(5, n_obs)

    if cv >= 3:
        scores = cross_val_score(pipeline, X_c, y_c, cv=cv, scoring="r2")
        r2_cv = scores.mean()
    else:
        r2_cv = np.nan

    pipeline.fit(X_c, y_c)
    y_pred = pipeline.predict(X_c)

    df_model.loc[df_c.index, "rendimiento_predicho"] = y_pred

    coef = pd.Series(
        pipeline.named_steps["model"].coef_,
        index=vars_indice
    )

    coeficientes[c] = coef

    metricas.append({
        "cluster": c,
        "n_obs": n_obs,
        "r2_cv": r2_cv,
        "r2_train": r2_score(y_c, y_pred),
        "mae_train": mean_absolute_error(y_c, y_pred),
        "rendimiento_promedio": y_c.mean(),
        "rendimiento_predicho_promedio": y_pred.mean()
    })

metricas_modelo = pd.DataFrame(metricas)
metricas_modelo


In [ ]:
coeficientes_df = (
    pd.DataFrame(coeficientes)
    .reset_index()
    .rename(columns={"index": "variable"})
)

coeficientes_df.to_csv(OUTPUT_COEF, index=False)
metricas_modelo.to_csv(OUTPUT_METRICAS, index=False)

print("Coeficientes guardados:", OUTPUT_COEF)
print("Métricas guardadas:", OUTPUT_METRICAS)


## 6. Construcción del índice climático calibrado

El índice se calcula por clúster:

1. Se estandarizan las variables dentro del clúster.
2. Se ponderan con los coeficientes Ridge del clúster.
3. Se normaliza el índice dentro del clúster.

**Interpretación:**  
Valores bajos del índice indican mayor riesgo climático relativo.


In [ ]:
df_model["indice_cluster_raw"] = np.nan

for c in sorted(df_model["cluster"].unique()):

    idx = df_model["cluster"] == c
    df_c = df_model.loc[idx].copy()

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_c[vars_indice])
    X_scaled = pd.DataFrame(X_scaled, columns=vars_indice, index=df_c.index)

    pesos = coeficientes[c].loc[vars_indice]
    indice_raw = X_scaled.mul(pesos, axis=1).sum(axis=1)

    df_model.loc[idx, "indice_cluster_raw"] = indice_raw

# Normalización dentro de cada clúster
df_model["indice_cluster"] = np.nan

for c in sorted(df_model["cluster"].unique()):

    idx = df_model["cluster"] == c

    media = df_model.loc[idx, "indice_cluster_raw"].mean()
    std = df_model.loc[idx, "indice_cluster_raw"].std()

    if std == 0 or pd.isna(std):
        df_model.loc[idx, "indice_cluster"] = 0
    else:
        df_model.loc[idx, "indice_cluster"] = (
            df_model.loc[idx, "indice_cluster_raw"] - media
        ) / std

df_model[["municipio", "anio", "cluster", TARGET, "rendimiento_predicho", "indice_cluster"]].head()


## 7. Validación visual del índice

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df_model["indice_cluster"], df_model[TARGET], alpha=0.75)
plt.xlabel("Índice climático calibrado")
plt.ylabel(TARGET)
plt.title("Índice climático vs rendimiento")
plt.grid(True)
plt.show()


In [ ]:
for c in sorted(df_model["cluster"].unique()):

    df_c = df_model[df_model["cluster"] == c]

    plt.figure(figsize=(5, 4))
    plt.scatter(df_c["indice_cluster"], df_c[TARGET], alpha=0.75)
    plt.xlabel("Índice climático calibrado")
    plt.ylabel(TARGET)
    plt.title(f"Cluster {c}")
    plt.grid(True)
    plt.show()


## 8. Evento de pérdida

Se define pérdida como rendimiento por debajo del percentil histórico seleccionado.

Este umbral se usa para calibrar el trigger del seguro.


In [ ]:
umbral_perdida = df_model[TARGET].quantile(LOSS_QUANTILE)

df_model["evento_perdida_global"] = (
    df_model[TARGET] <= umbral_perdida
).astype(int)

print("Umbral de pérdida:", round(umbral_perdida, 4))
print(df_model["evento_perdida_global"].value_counts())


## 9. Optimización del trigger

El trigger se elige minimizando una función de costo que penaliza más los falsos negativos.

- **Falso negativo:** hubo pérdida, pero el índice no activa pago.
- **Falso positivo:** no hubo pérdida, pero el índice activa pago.


In [ ]:
y_true = df_model["evento_perdida_global"]
indice = df_model["indice_cluster"]

umbrales = np.linspace(indice.min(), indice.max(), 200)
resultados = []

for t in umbrales:

    pred = (indice <= t).astype(int)

    TP = ((y_true == 1) & (pred == 1)).sum()
    TN = ((y_true == 0) & (pred == 0)).sum()
    FP = ((y_true == 0) & (pred == 1)).sum()
    FN = ((y_true == 1) & (pred == 0)).sum()

    TPR = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    FPR = FP / (FP + TN) if (FP + TN) > 0 else np.nan

    resultados.append({
        "threshold": t,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TPR": TPR,
        "FPR": FPR,
        "FN_rate": FN / len(df_model),
        "FP_rate": FP / len(df_model)
    })

df_eval = pd.DataFrame(resultados)
df_eval["score"] = df_eval["FN_rate"] * PENALTY_FN + df_eval["FP_rate"]
df_eval["youden"] = df_eval["TPR"] - df_eval["FPR"]

best = df_eval.loc[df_eval["score"].idxmin()]
trigger = best["threshold"]

auc_value = roc_auc_score(y_true, -indice)

print("Trigger óptimo:", round(trigger, 4))
print("AUC:", round(auc_value, 4))
print("FN_rate:", round(best["FN_rate"], 4))
print("FP_rate:", round(best["FP_rate"], 4))

df_eval.to_csv(OUTPUT_EVAL_TRIGGER, index=False)


In [ ]:
plt.figure(figsize=(6, 5))
plt.plot(df_eval["FPR"], df_eval["TPR"])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC - Índice climático | AUC={auc_value:.3f}")
plt.grid(True)
plt.show()


## 10. Función de pago

La función de pago usa:

- `trigger`: punto en el que empieza a pagar.
- `limite`: punto severo donde paga el 100%.
- `PAYOUT_EXPONENT`: suaviza o intensifica pagos intermedios.


In [ ]:
limite = df_model.loc[
    df_model["evento_perdida_global"] == 1,
    "indice_cluster"
].quantile(0.10)

# Validación para asegurar que el límite esté por debajo del trigger.
if limite >= trigger:
    limite = df_model["indice_cluster"].quantile(0.05)

print("Trigger:", round(trigger, 4))
print("Límite:", round(limite, 4))


In [ ]:
def calcular_payout(indice, trigger, limite, exponent=1.0):
    if indice >= trigger:
        return 0.0
    elif indice <= limite:
        return 1.0
    else:
        x = (trigger - indice) / (trigger - limite)
        return float(x ** exponent)


df_model["payout"] = df_model["indice_cluster"].apply(
    lambda x: calcular_payout(
        x,
        trigger=trigger,
        limite=limite,
        exponent=PAYOUT_EXPONENT
    )
)

df_model["payout"].describe()


## 11. Clasificación de riesgo

In [ ]:
# Riesgo alto: activa pago.
# Riesgo medio: no activa pago, pero se ubica en zona intermedia del índice.
# Riesgo bajo: índice alto, condiciones favorables.

umbral_medio = df_model["indice_cluster"].quantile(0.70)

if umbral_medio <= trigger:
    umbral_medio = df_model["indice_cluster"].quantile(0.80)

def clasificar_riesgo(x):
    if x <= trigger:
        return "Alto"
    elif x <= umbral_medio:
        return "Medio"
    else:
        return "Bajo"

df_model["riesgo"] = df_model["indice_cluster"].apply(clasificar_riesgo)

resumen_riesgo = (
    df_model
    .groupby("riesgo")
    .agg(
        n=("riesgo", "size"),
        rendimiento_promedio=(TARGET, "mean"),
        rendimiento_mediano=(TARGET, "median"),
        indice_promedio=("indice_cluster", "mean"),
        payout_promedio=("payout", "mean"),
        tasa_evento_perdida=("evento_perdida_global", "mean")
    )
    .reset_index()
)

resumen_riesgo


In [ ]:
orden = ["Alto", "Medio", "Bajo"]

resumen_plot = resumen_riesgo.set_index("riesgo").reindex(orden)

plt.figure(figsize=(6, 4))
plt.bar(resumen_plot.index, resumen_plot["rendimiento_promedio"])
plt.xlabel("Nivel de riesgo")
plt.ylabel("Rendimiento promedio")
plt.title("Rendimiento promedio por nivel de riesgo")
plt.grid(axis="y")
plt.show()


## 12. Evaluación del payout

In [ ]:
payout_perdida = df_model.loc[df_model["evento_perdida_global"] == 1, "payout"].mean()
payout_no_perdida = df_model.loc[df_model["evento_perdida_global"] == 0, "payout"].mean()
separacion_seguro = payout_perdida - payout_no_perdida

prima_pura = df_model["payout"].mean()
prima_comercial = prima_pura * (1 + MARGEN_COMERCIAL)

print("Payout promedio:", round(prima_pura, 4))
print("Payout en pérdida:", round(payout_perdida, 4))
print("Payout sin pérdida:", round(payout_no_perdida, 4))
print("Separación del seguro:", round(separacion_seguro, 4))
print("Prima pura:", round(prima_pura, 4))
print("Prima comercial:", round(prima_comercial, 4))


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df_model["payout"], df_model[TARGET], alpha=0.75)
plt.xlabel("Payout")
plt.ylabel(TARGET)
plt.title("Pago vs rendimiento")
plt.grid(True)
plt.show()


In [ ]:
df_model.boxplot(column="payout", by="evento_perdida_global", figsize=(6, 4))
plt.title("Distribución de pagos por evento de pérdida")
plt.suptitle("")
plt.xlabel("Evento de pérdida")
plt.ylabel("Payout")
plt.grid(True)
plt.show()


## 13. Guardar salidas del índice

Este bloque guarda los artefactos que serán usados por los siguientes notebooks:

- `06_simulacion_seguro.ipynb`
- `07_modelo_seguro_completo.ipynb`
- futura API


In [ ]:
# Guardar índice histórico
df_model.to_csv(OUTPUT_INDICE, index=False)
resumen_riesgo.to_csv(OUTPUT_RESUMEN_RIESGO, index=False)

# Guardar parámetros del seguro en formato ancho, fácil de consumir por API.
parametros_seguro = pd.DataFrame([{
    "version_modelo": "v1_indice_cluster_ridge",
    "fecha_calibracion": date.today().isoformat(),
    "target": TARGET,
    "n_observaciones": len(df_model),
    "n_municipios": df_model["municipio"].nunique(),
    "n_clusters": df_model["cluster"].nunique(),
    "n_variables_indice": len(vars_indice),
    "variables_indice": ";".join(vars_indice),
    "loss_quantile": LOSS_QUANTILE,
    "umbral_perdida_rendimiento": umbral_perdida,
    "trigger": trigger,
    "limite": limite,
    "umbral_medio_riesgo": umbral_medio,
    "penalty_fn": PENALTY_FN,
    "payout_exponent": PAYOUT_EXPONENT,
    "margen_comercial": MARGEN_COMERCIAL,
    "prima_pura": prima_pura,
    "prima_comercial": prima_comercial,
    "auc": auc_value,
    "payout_perdida": payout_perdida,
    "payout_no_perdida": payout_no_perdida,
    "separacion_seguro": separacion_seguro
}])

parametros_seguro.to_csv(OUTPUT_PARAMETROS, index=False)

print("Archivos guardados:")
print(OUTPUT_INDICE)
print(OUTPUT_RESUMEN_RIESGO)
print(OUTPUT_PARAMETROS)
print(OUTPUT_VARIABLES_INDICE)
print(OUTPUT_COEF)
print(OUTPUT_METRICAS)
print(OUTPUT_EVAL_TRIGGER)


## 14. Salidas generadas

Este notebook deja listos los siguientes artefactos:

### `data/model`
- `variables_indice_final.csv`
- `coeficientes_indice_por_cluster.csv`
- `metricas_modelo_indice_por_cluster.csv`
- `parametros_seguro.csv`

### `data/outputs`
- `indice_riesgo_climatico_2007-2024.csv`
- `resumen_riesgo_indice.csv`
- `evaluacion_trigger_indice.csv`

El siguiente paso del pipeline es:

```text
06_simulacion_seguro.ipynb
```

Ese notebook debe usar `indice_riesgo_climatico_2007-2024.csv` y `parametros_seguro.csv` para evaluar desempeño financiero, loss ratio y simulación Monte Carlo.
